# 00 — Groundwater Flow (GWF) Model

This notebook builds a MODFLOW 6 transient groundwater flow model for an open-cut mining scenario
using [flopy](https://flopy.readthedocs.io) as the Python interface.

---

## Set up

An open-cut mine is planned in a 2 km × 2 km unconfined aquifer. The pit (200 m × 200 m)
sits in the centre of the domain. To keep the pit dry during mining, groundwater must be
actively extracted. Some of that extracted water is re-injected to the west (managed aquifer
recharge, MAR) to limit regional drawdown and protect a groundwater-dependent ecosystem (GDE)
on the western boundary.

The simulation covers three phases:

| Period | Duration | Purpose |
|--------|----------|---------|
| 0 | 1 day | Establish steady-state pre-mining conditions |
| 1 | 10 years | Active dewatering + MAR |
| 2 | 100 years | Post-mining recovery |

---

## How this notebook is organised

Each MODFLOW 6 package is introduced in its own cell with a short explanation of what it does
and why the chosen parameters matter. Complex geometry and time-series utilities live in
`herebedragons.py` so they don't clutter the narrative here.

## Imports

We import `flopy` (the MODFLOW 6 interface), `numpy` (array construction), and a set of
project-specific helpers from `herebedragons.py` that handle the pit geometry and well setup.

In [ ]:
import os
import shutil
import flopy
import pyemu
import numpy as np

from herebedragons import (
    plot_setup, specify_pit_cells,
    build_wells_dewater, build_wells_mar, build_utlobs,
    prep_bins
)

## Model parameters

All key values are defined here so they are easy to find and experiment with.
Changing a value here automatically propagates to every package that uses it.

> **Try it:** double the pit size (`pit_side_length = 400`) or halve the hydraulic
> conductivity (`K = 2.5`) and re-run the notebook to see how drawdown changes.

In [ ]:
# Domain
Lx              = 2000      # model extent (m), square
delr            = 20        # cell size (m)
top             = 110       # surface elevation (m AHD)
botm            = 0         # aquifer base (m AHD)
pit_side_length = 200       # pit square side length (m)

# Hydraulic properties
K   = 5.0   # horizontal hydraulic conductivity (m/day)
Sy  = 0.05  # specific yield (dimensionless) — drainable porosity
Ss  = 1e-6  # specific storage (1/m) — elastic storage, minor here

# Boundary condition heads
gde_stage = 85   # drain elevation for the GDE (m AHD)
chd_stage = 95   # eastern inflow boundary head (m AHD)

# Stress periods
start_date_time  = '2025-01-01'
dewater_years    = 10
tr_perlen        = dewater_years * 365      # dewatering period length (days)
tr_tsps          = 12 * dewater_years       # ~monthly time steps during dewatering
recovery_years   = 100
recovery_perlen  = recovery_years * 365     # recovery period length (days)
recovery_tsps    = recovery_years           # annual time steps during recovery

# Well rates (positive = injection, negative = extraction)
dewater_rate = -800   # m³/day per dewatering well
mar_rate     =  800   # m³/day per MAR well

# Workspace — all MODFLOW input/output files go here
ws = 'model'

## Simulation and GWF model object

In MODFLOW 6, an `MFSimulation` is the top-level container that coordinates timing,
solvers, and one or more models. Here we attach a single groundwater flow (`GWF`) model.

Key options:
- **`continue_=True`** — if a time step fails to converge, MODFLOW writes a result anyway
  and keeps going rather than crashing.
- **`newtonoptions='UNDER_RELAXATION'`** — activates Newton–Raphson linearisation for the
  unconfined layer. Without this, the model can oscillate or diverge when cells go dry
  near the pit.
- **`save_flows=True`** — writes cell-by-cell flow terms to the budget file so we can
  later compute water balances.

In [ ]:
if os.path.exists(ws):
    shutil.rmtree(ws)
os.mkdir(ws)

sim = flopy.mf6.MFSimulation(sim_ws=ws, continue_=True)
gwf = flopy.mf6.ModflowGwf(sim, newtonoptions='UNDER_RELAXATION', save_flows=True)

## Time discretization (TDIS)

TDIS controls how the simulation is divided into **stress periods** and **time steps**.

- A **stress period** is a block of time during which boundary condition values are held
  constant (e.g. pumping rate, recharge rate). You can think of it as a "scenario phase".
- Within each stress period, MODFLOW subdivides time into **time steps** for numerical
  integration. More time steps → more accurate but slower.

Each row of `perioddata` is `(perlen, nstp, tsmult)` — period length, number of time steps,
and a multiplier on time-step length (1 = equal steps).

We use monthly steps during dewatering (fast head changes) and annual steps during recovery
(slow, diffuse rebound) to keep runtime reasonable without sacrificing accuracy.

In [ ]:
perioddata = [
    (1,               1,             1),  # period 0: 1-day steady-state warm-up
    (tr_perlen,       tr_tsps,       1),  # period 1: dewatering (monthly steps)
    (recovery_perlen, recovery_tsps, 1),  # period 2: recovery   (annual steps)
]
tdis = flopy.mf6.ModflowTdis(
    sim,
    nper=len(perioddata),
    perioddata=perioddata,
    time_units='days',
    start_date_time=start_date_time,
)

## Solver (IMS)

The Iterative Model Solution (IMS) is MODFLOW 6's linear–nonlinear solver. It uses a
two-level iteration scheme:

- **Outer iterations** (`outer_maximum`) — Newton nonlinear iterations that update the
  conductance matrix when heads change (needed for unconfined cells).
- **Inner iterations** (`inner_maximum`) — linear solver iterations (BICGSTAB here)
  that solve the matrix equation for a fixed conductance.

We use `complexity='COMPLEX'` which tells MODFLOW to use more aggressive preconditioning —
the default `SIMPLE` setting can struggle with the large head gradients near the pit.
`BICGSTAB` (biconjugate gradient stabilised) generally outperforms the default `CG`
for asymmetric problems like this one.

`dvclose` is the head-change convergence criterion (m). `rcloserecord = '1 strict'` means
the residual tolerance is 1 m³/day and *all* cells must satisfy it (strict mode).

Finally, the IMS must be registered to the GWF model.

In [ ]:
ims = flopy.mf6.ModflowIms(
    sim,
    complexity='COMPLEX',
    linear_acceleration='BICGSTAB',
    outer_maximum=1000,
    outer_dvclose=0.001,
    inner_maximum=100,
    inner_dvclose=0.001,
    rcloserecord='1 strict',
    reordering_method='none',
    no_ptcrecord='FIRST',
)
sim.register_ims_package(ims, [gwf.name])

## Grid (DIS)

DIS defines the spatial discretization — how the aquifer volume is divided into cells.
We use a **structured rectilinear grid**: uniform rows and columns, a single layer.

- 100 × 100 cells at 20 m spacing → 2000 m × 2000 m domain
- Single layer from 0 to 110 m elevation (110 m thick)
- `idomain = 1` everywhere — all cells are active
  (the pit cells will be tagged separately as `idomain = 2` for bookkeeping)

`set_all_data_external()` writes the array data to separate matrix-like text files rather than
embedding them in the text input files — keeps files readable and speeds up writes.

In [ ]:
nlay = 1
nrow = int(Lx / delr)   # 100 rows
ncol = int(Lx / delr)   # 100 columns

dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay, nrow=nrow, ncol=ncol,
    delr=delr, delc=delr,
    top=top  * np.ones((nrow, ncol)),
    botm=botm * np.ones((nlay, nrow, ncol)),
    idomain=np.ones((nlay, nrow, ncol), dtype=int),
    length_units='meters',
)
dis.set_all_data_external()

## Hydraulic properties (NPF)

The Node Property Flow (NPF) package assigns hydraulic conductivity to each cell.

- `K = 5 m/day` — a moderately transmissive sandy aquifer. With 110 m saturated thickness
  this gives a transmissivity of ~550 m²/day.
- `icelltype = 1` — **convertible** (unconfined) layer. MODFLOW recalculates saturated
  thickness each iteration as heads change. Set to `0` for a confined layer where
  transmissivity is constant.

The conductivity is spatially uniform here, but `k_array` is a full 3D numpy array so you
can easily substitute a heterogeneous field (e.g. from a kriging output).

In [ ]:
k_array = K * np.ones_like(gwf.dis.idomain.get_data(), dtype=float)
npf = flopy.mf6.ModflowGwfnpf(gwf, k=k_array, icelltype=1)
npf.set_all_data_external()

## Storage (STO)

STO governs how much water is released or taken up per unit head change per unit volume.

- **Specific yield (`Sy = 0.05`)** — the fraction of the aquifer volume that drains under
  gravity when the water table drops. Relevant for unconfined (water-table) conditions.
  A sandy material might be 0.1–0.3; we use 0.05 for a silty sand.
- **Specific storage (`Ss = 1×10⁻⁶ /m`)** — elastic compressibility of the aquifer matrix
  and water. Negligible for unconfined flow but required by MODFLOW.
- **`iconvert = 1`** — cells switch between confined and unconfined storage depending on
  whether the head is above or below the cell top. Required when `icelltype=1` in NPF.

Period 0 is declared steady-state (`steady_state={0: True}`) so MODFLOW solves for an
equilibrium head distribution without any storage terms — this gives us a clean
pre-mining baseline. All subsequent periods are transient.

In [ ]:
shape    = gwf.dis.idomain.get_data().shape
sy_array = Sy * np.ones(shape, dtype=float)
ss_array = Ss * np.ones(shape, dtype=float)

sto = flopy.mf6.ModflowGwfsto(
    gwf,
    sy=sy_array, ss=ss_array, iconvert=1,
    steady_state={0: True},
    transient={1: True},
    save_flows=True,
)
sto.set_all_data_external()

## Initial conditions (IC)

IC sets the starting head in every cell at the beginning of the simulation.

Because period 0 is steady-state, the exact starting values do not matter — the solver will
find the correct equilibrium regardless. Setting heads equal to the surface elevation (`top`)
is a common safe choice: all cells start saturated, so no cells go dry before the solver
has a chance to establish the water table.

In [ ]:
strt = gwf.dis.nlay.get_data() * [gwf.dis.top.get_data()]
ic = flopy.mf6.ModflowGwfic(gwf, strt=strt)

## Output control (OC)

OC specifies what MODFLOW writes to disk and when.

- **`head_filerecord`** — heads are saved to a binary `.hds` file every time step.
  flopy's `HeadFile` utility reads this for post-processing.
- **`saverecord=[('HEAD', 'ALL')]`** — save heads at the end of every time step.
  You can change `'ALL'` to `'LAST'` to save only the final time step (smaller file).
- **`printrecord=[('BUDGET', 'ALL')]`** — write the volumetric budget to the listing file
  every time step. Useful for checking that the model is mass-conservative.

In [ ]:
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f'{gwf.name}.hds',
    saverecord=[('HEAD', 'ALL')],
    printrecord=[('BUDGET', 'ALL')],
)

## Recharge (RCH)

RCH applies a spatially uniform diffuse recharge to the top of the model, representing
rainfall infiltration. The rate of 1×10⁻⁵ m/day is approximately 3.6 mm/year — a very
arid climate scenario. It is small enough that recharge has negligible influence on pit
dewatering at this scale, but it is included for completeness and to prevent cells from
going completely dry in the far field during recovery.

`ModflowGwfrcha` (the 'a' suffix) applies recharge to the highest active cell in each
column — correct for an unconfined aquifer where the water table moves.

In [ ]:
rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=1e-5)
rch.set_all_data_external()

## Lateral inflow boundary (GHB)

The right (eastern) column of the model is a **General-Head Boundary** — a flux boundary
that lets regional groundwater flow into the model domain.

The GHB flux is proportional to the head difference between the aquifer and the external
head (`chd_stage = 95 m`), scaled by the conductance (C, m²/day):

$$Q = C \cdot (h_{\text{boundary}} - h_{\text{cell}})$$

This is more realistic than a fixed-head (CHD) boundary because the inflow reduces
naturally as the aquifer head rises toward the boundary head — it cannot supply infinite
water. A conductance of 100 m²/day per cell is moderate; you can think of it as a leaky
connection to a large regional aquifer.

The boundary is active in every stress period.

In [ ]:
cond_ghb = 100   # conductance (m²/day)
ghb_j    = gwf.dis.ncol.get_data() - 1   # last column index
nrow_ghb = gwf.dis.nrow.get_data()
nper     = sim.tdis.nper.get_data()

ghb_spd_i = [((0, i, ghb_j), chd_stage, cond_ghb) for i in range(nrow_ghb)]
ghb_spd   = {kper: ghb_spd_i for kper in range(nper)}

ghb = flopy.mf6.ModflowGwfghb(
    gwf,
    stress_period_data=ghb_spd,
    pname='ghb-inflow',
    filename='inflow.ghb',
)
ghb.set_all_data_external()

## Pit geometry and dewatering wells (WEL)

### Pit cells

`specify_pit_cells` uses a polygon–grid intersection to identify all model cells that fall
inside the pit footprint. These cells are tagged with `idomain = 2` (a user-defined zone)
purely for bookkeeping. Makes it easy to
query pit-cell heads later.

### Dewatering wells

Rather than pumping from inside the pit (where the aquifer may go dry), wells are placed
in the **ring of cells immediately surrounding** the pit perimeter. This mimics real pit
dewatering practice; a series of production bores around the pit border.

Each well uses a **stepwise time series** so MODFLOW interpolates the rate:

| Time (days) | Rate (m³/day) |
|-------------|---------------|
| 0           | 0 (off) |
| 1           | `dewater_rate` (on) |
| 3650        | 0 (off — mining finished) |

`auto_flow_reduce=0.1` prevents cells from going dry by automatically reducing the
extraction rate when the available water is less than 10% of the requested rate.

In [ ]:
pitcells    = specify_pit_cells(gwf, pit_side_length)
wel_dewater = build_wells_dewater(sim, pitcells, rate=dewater_rate)

## Managed aquifer recharge wells (WEL)

MAR wells reinject some of the extracted water into the aquifer to the west of the pit.
The goals are:

1. **Reduce net drawdown** — recharge partially offsets the regional cone of depression.
2. **Protect the GDE** — the drain on the western edge is sensitive to water-table decline;
   injecting nearby maintains heads.
3. **Accelerate recovery** — recharged water is available for natural recovery after mining.

Wells are spaced every 10 rows (200 m apart) in a column at `mar_j` — roughly halfway
between the pit and the western boundary. They use the same stepwise time series as the
dewatering wells, so injection runs only during the active mining period.

In [ ]:
wel_mar = build_wells_mar(sim, pitcells, rate=mar_rate)

## Groundwater-dependent ecosystem drain (DRN)

The western column of cells represents a GDE — vegetation whose roots access the
water table. The **Drain (DRN)** package models the GDE as a linear seepage face:
when the water table rises above the drain elevation (`gde_stage = 85 m`), water
discharges at a rate proportional to the head above the drain. When the water table
is below `gde_stage`, the drain is inactive (it cannot inject water — that is a key
distinction from GHB).

$$Q_{\text{drain}} = \begin{cases} C(h - h_{\text{drain}}) & \text{if } h > h_{\text{drain}} \\ 0 & \text{otherwise} \end{cases}$$

We also attach a **drain observation** (`.obs`) to record the total drain flux over time.
This lets us quantify how much water the GDE receives during and after dewatering — a
key environmental compliance metric.

In [ ]:
cond_drn = 100   # conductance (m²/day)
nrow_drn = gwf.dis.nrow.get_data()
nper     = sim.tdis.nper.get_data()

drn_spd_i = [((0, row, 0), gde_stage, cond_drn, 'drn-gde') for row in range(nrow_drn)]
drn_spd   = {kper: drn_spd_i for kper in range(nper)}

drn = flopy.mf6.ModflowGwfdrn(
    gwf,
    stress_period_data=drn_spd,
    pname='drn-gde',
    boundnames=True,
)
drn.set_all_data_external()

# Record total drain flux to CSV for post-processing
drn_obs = {f'{gwf.name}.obs.drn.csv': [('drn-gde', 'DRN', 'drn-gde')]}
drn.obs.initialize(digits=10, print_input=False, continuous=drn_obs)

## Head observations (OBS)

MODFLOW's OBS utility writes simulated heads at specified locations to CSV files every
time step. We record heads at:

- **Every pit cell** — to track when the water table falls below the pit floor during
  dewatering and when it recovers afterwards.
- **Every MAR well cell** — to check that injection does not cause upwelling above ground
  surface (head > `top = 110 m`).

These CSV files are also the primary input for model calibration — simulated heads at
observation locations are compared to measured values to estimate aquifer parameters.

In [ ]:
utlobs = build_utlobs(gwf, pitcells)

# Prepare executables (bins)

We need to copy the right executables into the model directory

In [ ]:
prep_bins(ws)

## Write and run

`sim.write_simulation()` writes all MODFLOW 6 input files to the `model/` directory.
You can inspect them as plain text (`.nam`, `.tdis`, `.dis`, etc.) to understand the
native MODFLOW 6 file format.

In [ ]:
sim.set_all_data_external()
sim.write_simulation()

`pyemu.os_utils.run("mf6", cwd=ws)` launches the MODFLOW 6 executable and streams the listing output.

In [ ]:
pyemu.os_utils.run("mf6", cwd=ws)

## Visualise model setup

A map-view plot of the model grid showing all boundary condition locations and the pit
outline. The figure is also saved to `model_setup.pdf`.

Check that:
- Dewatering wells (red) form a complete ring around the pit outline
- MAR wells (black) are in a column to the west
- The GDE drain (cyan) lines the left edge
- The GHB inflow (blue) lines the right edge

In [ ]:
plot_setup(gwf, pitcells)